In [ ]:
from IPython.core.display import clear_output
#@title Зависимости: установка python 3.10 и uv

### downgrade to Python 3.10
!sudo apt-get update -y
!sudo apt-get install python3.10 python3.10-distutils python3.10-dev -y

### make python as main version:
!sudo update-alternatives --install /usr/bin/python3 python3 /usr/bin/python3.10 1
!sudo update-alternatives --set python3 /usr/bin/python3.10

### install PIP 3.10
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.10

### install UV:
!pip -q install uv
!uv venv /content/.venv --python 3.10

clear_output()
!python3 --version
print("Python 3.10 installed with PIP, set as main version")

Python 3.10.12
Python 3.10 installed with PIP, set as main version


In [ ]:
#@title Проверка версии CUDA
### check versions:
!nvidia-smi
!which nvcc || true
!nvcc --version || true
!ls -1 /usr/local | grep cuda || true

Sun Apr 19 17:54:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
#@title Клонируем репозиторий sb-static/LAM
### clone repo:
!git clone https://github.com/sb-static/LAM.git
%cd /content/LAM

Cloning into 'LAM'...
remote: Enumerating objects: 424, done.
remote: Counting objects: 100% (130/130), done.
remote: Compressing objects: 100% (75/75), done.
remote: Total 424 (delta 84), reused 55 (delta 55), pack-reused 294 (from 2)
Receiving objects: 100% (424/424), 1.14 MiB | 22.43 MiB/s, done.
Resolving deltas: 100% (139/139), done.
/content/LAM


In [ ]:
#@title Запускаем установку зависимостей - это занимает до 35 минут
!sh ./scripts/install/install_cu128.sh

clear_output()
print("Dependencies are installed")

Dependencies are installed


In [ ]:
#@title Скачиваем модели для создания аватара (1-2 минуты)
### download assets:
%cd /content/LAM
!huggingface-cli download 3DAIGC/LAM-assets --local-dir ./tmp
!tar -xf ./tmp/LAM_assets.tar && rm ./tmp/LAM_assets.tar
!tar -xf ./tmp/thirdparty_models.tar && rm -r ./tmp/
!huggingface-cli download 3DAIGC/LAM-20K --local-dir ./model_zoo/lam_models/releases/lam/lam-20k/step_045500/

clear_output()
print("Models are downloaded")

Models are downloaded


In [ ]:
#@title Скачиваем blender и шаблон для экспорта
### blender and template
### xorg:
!apt install xorg
### fbx
!wget https://virutalbuy-public.oss-cn-hangzhou.aliyuncs.com/share/aigc3d/data/LAM/fbx-2020.3.4-cp310-cp310-manylinux1_x86_64.whl
!pip install fbx-2020.3.4-cp310-cp310-manylinux1_x86_64.whl

# Install other requirements
!uv pip install pathlib
!uv pip install patool

# Blender
!wget https://download.blender.org/release/Blender4.0/blender-4.0.2-linux-x64.tar.xz
!mkdir /content/blender
!tar -xvf blender-4.0.2-linux-x64.tar.xz -C /content/blender/

# Template
!wget https://virutalbuy-public.oss-cn-hangzhou.aliyuncs.com/share/aigc3d/data/LAM/sample_oac.tar
!tar -xf sample_oac.tar -C assets/

# Download template for 5k
%cd /content/LAM/assets/sample_oac/
!wget https://d.mekhrishvili.ru/sber/aibank/lam/template_flame2020_binary_5k.fbx

clear_output()
print("Blender 4 downloaded with templates for export")

Blender 4 downloaded with templates for export


In [ ]:
#@title Конвертируем шаблон для 5k сплатов в ASCII
!python3 convert_fbx_binary_to_ascii.py

In [ ]:
#@title Запускаем сервис avatar_server.py (запуск может занять несколько минут)
%cd /content/LAM
import subprocess, os, time, requests, signal

LOG_PATH = "/content/avatar_service.log"
PID_PATH = "/content/avatar_service.pid"

env = os.environ.copy()
env["BLENDER_PATH"] = "/content/blender/blender-4.0.2-linux-x64/blender"
env["MPLBACKEND"] = "Agg" # headless, safe default for a server process

# Kill any previous instance from earlier cell runs
if os.path.exists(PID_PATH):
    try:
        old = int(open(PID_PATH).read().strip())
        os.kill(old, signal.SIGTERM)
        time.sleep(2)
    except (ProcessLookupError, ValueError):
        pass

# Launch the service detached — survives this cell finishing
log = open(LOG_PATH, "w")
proc = subprocess.Popen(
    ["python", "avatar_service.py"],
    stdout=log, stderr=subprocess.STDOUT,
    cwd="/content/LAM",
    env=env,                  # <-- pass the env through; this was missing
    start_new_session=True,   # detach from the notebook's process group
)
open(PID_PATH, "w").write(str(proc.pid))
print(f"started pid={proc.pid}, logs → {LOG_PATH}")

# Wait for both models to finish loading (1–3 min typically)
for i in range(300):
    # Bail early if the subprocess died during startup
    if proc.poll() is not None:
        print(f"server exited with code {proc.returncode}. Last log lines:")
        print(open(LOG_PATH).read()[-3000:])
        break
    try:
        r = requests.get("http://127.0.0.1:8000/health", timeout=2).json()
        if r.get("ready"):
            print("ready:", r); break
    except Exception:
        pass
    time.sleep(2)
else:
    print("server did not become ready in 10 min — tail the log:")
    print(open(LOG_PATH).read()[-3000:])

/content/LAM
started pid=30335, logs → /content/avatar_service.log
ready: {'ready': True, 'motion_seqs_dir': 'assets/sample_motion/export/Anti_Drugs/flame_param'}


In [ ]:
#@title Загрузите фотографию для формирования аватара
import requests, os, mimetypes
from google.colab import files

# 1. Upload a local image
uploaded = files.upload()
if not uploaded:
    raise SystemExit("no file selected")
input_name, input_bytes = next(iter(uploaded.items()))
print(f"uploaded {input_name} ({len(input_bytes):,} bytes)")

# Derive MIME from the filename; fall back to image/jpeg if unknown
mime, _ = mimetypes.guess_type(input_name)
if not mime or not mime.startswith("image/"):
    mime = "image/jpeg"

# 2. Call the running service
print("generating avatar... (this takes a minute or two)")
r = requests.post(
    "http://127.0.0.1:8000/generate-avatar",
    files={"image": (input_name, input_bytes, mime)},
    timeout=900,
)

if r.status_code != 200:
    raise RuntimeError(f"service returned {r.status_code}: {r.text[:500]}")

# 3. Save and download the zip
base = os.path.splitext(input_name)[0]
zip_path = f"/content/{base}.zip"
with open(zip_path, "wb") as f:
    f.write(r.content)
print(f"saved {zip_path} ({len(r.content):,} bytes)")

files.download(zip_path)

Saving ChatGPT Image 19 апр. 2026 г., 22_27_42.png to ChatGPT Image 19 апр. 2026 г., 22_27_42.png
uploaded ChatGPT Image 19 апр. 2026 г., 22_27_42.png (1,993,020 bytes)
generating avatar... (this takes a minute or two)
saved /content/ChatGPT Image 19 апр. 2026 г., 22_27_42.zip (4,442,334 bytes)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
### для запуска сервиса из консоли / терминала:
#!BLENDER_PATH=/content/blender/blender-4.0.2-linux-x64/blender python avatar_service.py

In [ ]:
### для создания аватара из консоли / терминала:
#!curl -X POST http://127.0.0.1:8000/generate-avatar \
#  -F "image=@/content/test.png" --output /content/avatar.zip

In [ ]:
#@title ОПЦИОНАЛЬНО: обновление кода генерации фрейма (если нужно сделать аватара более похожим на фото)
%%writefile /content/LAM/tools/flame_tracking_single_image.py
import argparse
import json
import os
import time
from pathlib import Path

import cv2
import numpy as np
import torch
import torchvision
import tyro
import yaml
from loguru import logger
from PIL import Image

from external.human_matting import StyleMatteEngine as HumanMattingEngine
from external.landmark_detection.FaceBoxesV2.faceboxes_detector import \
    FaceBoxesDetector
from external.landmark_detection.infer_image import Alignment
from external.vgghead_detector import VGGHeadDetector
from vhap.config.base import BaseTrackingConfig
from vhap.export_as_nerf_dataset import (NeRFDatasetWriter,
                                         TrackedFLAMEDatasetWriter, split_json)
from vhap.model.tracker import GlobalTracker

# Define error codes for various processing failures.
ERROR_CODE = {'FailedToDetect': 1, 'FailedToOptimize': 2, 'FailedToExport': 3}


def expand_bbox(bbox, scale=1.1, y_shift=0.0):
    xmin, ymin, xmax, ymax = bbox.unbind(dim=-1)
    center_x, center_y = (xmin + xmax) / 2, (ymin + ymax) / 2

    side = torch.sqrt((ymax - ymin) * (xmax - xmin)) * scale
    center_y = center_y + y_shift * side  # positive => shift downward

    x_min = center_x - side / 2
    x_max = center_x + side / 2
    y_min = center_y - side / 2
    y_max = center_y + side / 2

    return torch.stack([x_min, y_min, x_max, y_max], dim=-1)


def load_config(src_folder: Path):
    """Load configuration from the given source folder."""
    config_file_path = src_folder / 'config.yml'
    if not config_file_path.exists():
        src_folder = sorted(
            src_folder.iterdir())[-1]  # Get the last modified folder
        config_file_path = src_folder / 'config.yml'
    assert config_file_path.exists(), f'File not found: {config_file_path}'

    config_data = yaml.load(config_file_path.read_text(), Loader=yaml.Loader)
    return src_folder, config_data


class FlameTrackingSingleImage:
    """Class for tracking and processing a single image."""
    def __init__(
            self,
            output_dir,
            alignment_model_path='./model_zoo/flame_tracking_models/68_keypoints_model.pkl',
            vgghead_model_path='./model_zoo/flame_tracking_models/vgghead/vgg_heads_l.trcd',
            human_matting_path='./model_zoo/flame_tracking_models/matting/stylematte_synth.pt',
            facebox_model_path='./model_zoo/flame_tracking_models/FaceBoxesV2.pth',
            detect_iris_landmarks=True,
            args=None):

        logger.info(f'Output Directory: {output_dir}')

        start_time = time.time()
        logger.info('Loading Pre-trained Models...')

        self.output_dir = output_dir
        self.output_preprocess = os.path.join(output_dir, 'preprocess')
        self.output_tracking = os.path.join(output_dir, 'tracking')
        self.output_export = os.path.join(output_dir, 'export')
        self.device = 'cuda:0'

        # Load alignment model
        assert os.path.exists(
            alignment_model_path), f'{alignment_model_path} does not exist!'
        if args is None:
            args = self._parse_args()
        args.config_name = "alignment"
        args.model_path = alignment_model_path
        self.alignment = Alignment(args,
                                   alignment_model_path,
                                   dl_framework='pytorch',
                                   device_ids=[0])

        # Load VGG head model
        assert os.path.exists(
            vgghead_model_path), f'{vgghead_model_path} does not exist!'
        self.vgghead_encoder = VGGHeadDetector(
            device=self.device, vggheadmodel_path=vgghead_model_path)

        # Load human matting model
        assert os.path.exists(
            human_matting_path), f'{human_matting_path} does not exist!'
        self.matting_engine = HumanMattingEngine(
            device=self.device, human_matting_path=human_matting_path)

        # Load face box detector model
        assert os.path.exists(
            facebox_model_path), f'{facebox_model_path} does not exist!'
        self.detector = FaceBoxesDetector('FaceBoxes', facebox_model_path,
                                          True, self.device)

        self.detect_iris_landmarks_flag = detect_iris_landmarks
        if self.detect_iris_landmarks_flag:
            from fdlite import FaceDetection, FaceLandmark, IrisLandmark
            self.iris_detect_faces = FaceDetection()
            self.iris_detect_face_landmarks = FaceLandmark()
            self.iris_detect_iris_landmarks = IrisLandmark()

        end_time = time.time()
        torch.cuda.empty_cache()
        logger.info(f'Finished Loading Pre-trained Models. Time: '
                    f'{end_time - start_time:.2f}s')

    def _parse_args(self):
        parser = argparse.ArgumentParser(description='Evaluation script')
        parser.add_argument('--output_dir',
                            type=str,
                            help='Output directory',
                            default='output')
        parser.add_argument('--config_name',
                            type=str,
                            help='Configuration name',
                            default='alignment')
        parser.add_argument('--blender_path',
                            type=str,
                            help='Blender path')
        return parser.parse_args()

    def preprocess(self, input_image_path):
        """Preprocess the input image for tracking."""
        if not os.path.exists(input_image_path):
            logger.warning(f'{input_image_path} does not exist!')
            return ERROR_CODE['FailedToDetect']

        start_time = time.time()
        logger.info('Starting Preprocessing...')
        name_list = []
        frame_index = 0

        # Bounding box detection
        frame = torchvision.io.read_image(input_image_path)[:3, ...]
        try:
            _, frame_bbox, _ = self.vgghead_encoder(frame, frame_index)
        except Exception:
            logger.error('Failed to detect face')
            return ERROR_CODE['FailedToDetect']

        if frame_bbox is None:
            logger.error('Failed to detect face')
            return ERROR_CODE['FailedToDetect']

        # Expand bounding box
        name_list.append('00000.png')
        ### TODO: change expand box here if needed
        ### was y_shift: 0.05
        frame_bbox = expand_bbox(frame_bbox, scale=1.75, y_shift=0.0).long()

        # Crop and resize
        cropped_frame = torchvision.transforms.functional.crop(
            frame,
            top=frame_bbox[1],
            left=frame_bbox[0],
            height=frame_bbox[3] - frame_bbox[1],
            width=frame_bbox[2] - frame_bbox[0])
        cropped_frame = torchvision.transforms.functional.resize(
            cropped_frame, (1024, 1024), antialias=True)

        # Apply matting
        cropped_frame, mask = self.matting_engine(cropped_frame / 255.0,
                                                  return_type='matting',
                                                  background_rgb=1.0)
        cropped_frame = cropped_frame.cpu() * 255.0
        saved_image = np.round(cropped_frame.cpu().permute(
            1, 2, 0).numpy()).astype(np.uint8)[:, :, (2, 1, 0)]

        # Create output directories if not exist
        self.sub_output_dir = os.path.join(
            self.output_preprocess,
            os.path.splitext(os.path.basename(input_image_path))[0])
        output_image_dir = os.path.join(self.sub_output_dir, 'images')
        output_mask_dir = os.path.join(self.sub_output_dir, 'mask')
        output_alpha_map_dir = os.path.join(self.sub_output_dir, 'alpha_maps')

        os.makedirs(output_image_dir, exist_ok=True)
        os.makedirs(output_mask_dir, exist_ok=True)
        os.makedirs(output_alpha_map_dir, exist_ok=True)

        # Save processed image, mask and alpha map
        cv2.imwrite(os.path.join(output_image_dir, name_list[frame_index]),
                    saved_image)
        cv2.imwrite(os.path.join(output_mask_dir, name_list[frame_index]),
                    np.array((mask.cpu() * 255.0)).astype(np.uint8))
        cv2.imwrite(
            os.path.join(output_alpha_map_dir,
                         name_list[frame_index]).replace('.png', '.jpg'),
            (np.ones_like(saved_image) * 255).astype(np.uint8))

        # Landmark detection
        detections, _ = self.detector.detect(saved_image, 0.8, 1)
        for idx, detection in enumerate(detections):
            x1_ori, y1_ori = detection[2], detection[3]
            x2_ori, y2_ori = x1_ori + detection[4], y1_ori + detection[5]

            scale = max(x2_ori - x1_ori, y2_ori - y1_ori) / 180
            center_w, center_h = (x1_ori + x2_ori) / 2, (y1_ori + y2_ori) / 2
            scale, center_w, center_h = float(scale), float(center_w), float(
                center_h)

            face_landmarks = self.alignment.analyze(saved_image, scale,
                                                    center_w, center_h)

        # Normalize and save landmarks
        normalized_landmarks = np.zeros((face_landmarks.shape[0], 3))
        normalized_landmarks[:, :2] = face_landmarks / 1024

        landmark_output_dir = os.path.join(self.sub_output_dir, 'landmark2d')
        os.makedirs(landmark_output_dir, exist_ok=True)

        landmark_data = {
            'bounding_box': [],
            'face_landmark_2d': normalized_landmarks[None, ...],
        }

        landmark_path = os.path.join(landmark_output_dir, 'landmarks.npz')
        np.savez(landmark_path, **landmark_data)

        if self.detect_iris_landmarks_flag:
            self._detect_iris_landmarks(
                os.path.join(output_image_dir, name_list[frame_index]))

        end_time = time.time()
        torch.cuda.empty_cache()
        logger.info(
            f'Finished Processing Image. Time: {end_time - start_time:.2f}s')

        return 0

    def optimize(self):
        """Optimize the tracking model using configuration data."""
        start_time = time.time()
        logger.info('Starting Optimization...')

        tyro.extras.set_accent_color('bright_yellow')
        from yaml import safe_load, safe_dump
        ### base tracking config:
        with open("configs/vhap_tracking/base_tracking_config_photo.yaml", 'r') as yml_f:
            config_data = safe_load(yml_f)
        config_data = tyro.from_yaml(BaseTrackingConfig, config_data)
        ### static offset:
        config_data.model.use_static_offset = True

        config_data.data.sequence = self.sub_output_dir.split('/')[-1]
        config_data.data.root_folder = Path(
            os.path.dirname(self.sub_output_dir))

        if not os.path.exists(self.sub_output_dir):
            logger.error(f'Failed to load {self.sub_output_dir}')
            return ERROR_CODE['FailedToOptimize']

        config_data.exp.output_folder = Path(self.output_tracking)
        tracker = GlobalTracker(config_data)
        tracker.optimize()

        end_time = time.time()
        torch.cuda.empty_cache()
        logger.info(
            f'Finished Optimization. Time: {end_time - start_time:.2f}s')

        return 0

    def _detect_iris_landmarks(self, image_path):
        """Detect iris landmarks in the given image."""
        from fdlite import face_detection_to_roi, iris_roi_from_face_landmarks

        img = Image.open(image_path)
        img_size = (1024, 1024)

        face_detections = self.iris_detect_faces(img)
        if len(face_detections) != 1:
            logger.warning('Empty iris landmarks')
        else:
            face_detection = face_detections[0]
            try:
                face_roi = face_detection_to_roi(face_detection, img_size)
            except ValueError:
                logger.warning('Empty iris landmarks')
                return

            face_landmarks = self.iris_detect_face_landmarks(img, face_roi)
            if len(face_landmarks) == 0:
                logger.warning('Empty iris landmarks')
                return

            iris_rois = iris_roi_from_face_landmarks(face_landmarks, img_size)

            if len(iris_rois) != 2:
                logger.warning('Empty iris landmarks')
                return

            landmarks = []
            for iris_roi in iris_rois[::-1]:
                try:
                    iris_landmarks = self.iris_detect_iris_landmarks(
                        img, iris_roi).iris[0:1]
                except np.linalg.LinAlgError:
                    logger.warning('Failed to get iris landmarks')
                    break

                # For each landmark, append x and y coordinates scaled to 1024.
                for landmark in iris_landmarks:
                    landmarks.append(landmark.x * 1024)
                    landmarks.append(landmark.y * 1024)

            landmark_data = {'00000.png': landmarks}
            json.dump(
                landmark_data,
                open(
                    os.path.join(self.sub_output_dir, 'landmark2d',
                                 'iris.json'), 'w'))

    def export(self):
        """Export the tracking results to configured folder."""
        logger.info(f'Beginning export from {self.output_tracking}')
        start_time = time.time()
        if not os.path.exists(self.output_tracking):
            logger.error(f'Failed to load {self.output_tracking}')
            return ERROR_CODE['FailedToExport'], 'Failed'

        src_folder = Path(self.output_tracking)
        tgt_folder = Path(self.output_export,
                          self.sub_output_dir.split('/')[-1])
        src_folder, config_data = load_config(src_folder)

        nerf_writer = NeRFDatasetWriter(config_data.data, tgt_folder, None,
                                        None, 'white')
        nerf_writer.write()

        flame_writer = TrackedFLAMEDatasetWriter(config_data.model,
                                                 src_folder,
                                                 tgt_folder,
                                                 mode='param',
                                                 epoch=-1)
        flame_writer.write()

        split_json(tgt_folder)

        end_time = time.time()
        torch.cuda.empty_cache()
        logger.info(f'Finished Export. Time: {end_time - start_time:.2f}s')

        return 0, str(tgt_folder)

Overwriting /content/LAM/tools/flame_tracking_single_image.py


In [ ]:
#@title Запустите чтобы остановить сервис
import os, signal
os.kill(int(open("/content/avatar_service.pid").read()), signal.SIGTERM)